# ML-07 — Baseline Action Score and Top-10 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Skills loaded for this task: `building-baselines` + `flyrank/flyrank-data`. Lane 2 — Refresh /
Content Opportunity Scoring (locked, same lane as W01–W04). Data: the starter CSV
(`data/raw/content_refresh_anonymized.csv`), same slice the reference pipeline
(`scripts/01-02`) uses — this baseline is what my Week-5 model has to beat.

Note on the skeleton header: this repo's generic skill suggests a top-20 hand review, but this
assignment's card explicitly asks for **top-10** ("what done looks like: ten reviewed rows") —
that's what §3 does.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

if Path.cwd().name == "notebooks":
    import os
    os.chdir("../..")

sys.path.insert(0, "scripts")
from ml_utils import precision_at_k  # reference pipeline helper, imported not edited

RAW_PATH = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(RAW_PATH)

# Same lane-2 slice as scripts/01_prepare_features.py: visible pages old enough to review.
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# is_declining_label is used ONLY below to audit signals and sanity-check the queue.
# It is never a scoring input -- trend_direction / trend_pct are label-derived (flyrank-data
# skill), and this card explicitly requires "no future-window or label-derived inputs."
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Lane slice: {len(df):,} pages across {df['client_id'].nunique()} clients")
print(f"Base rate (declining proxy): {df['is_declining_label'].mean():.3f}")


Lane slice: 30,000 pages across 32 clients
Base rate (declining proxy): 0.542


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Two signals my rule idea leans on, checked first — one is flag-linked:**

- **Staleness** (`days_since_last_update >= 180`) — the signal behind FlyRank's live refresh
  flags (per the session and `docs/ml-intern-dataset-and-lane-guide.md`: those flags — `health_score`,
  `needs_ctr_fix`, `is_quick_win` — aren't shipped in this data, so I check the observable signal
  a refresh flag would lean on).
- **CTR vs. position** — the signal behind the CTR-fix logic (`needs_ctr_fix`): pages should get
  more clicks the higher they rank, so a page with unusually low CTR *for its position* is a real,
  fixable anomaly (title/meta/snippet), not noise.

Bucket tables for both, with `n`, below — then a one-word verdict each.


In [2]:
# --- Signal 1: staleness -> decline rate (behind the refresh flag) ---
df["stale_bucket"] = np.where(df["days_since_last_update"] >= 180, "stale_180plus", "fresh_under_180")
signal1 = df.groupby("stale_bucket")["is_declining_label"].agg(n="size", decline_rate="mean")
print("Signal 1 — staleness (>=180d since update) vs. decline rate:")
print(signal1)
print()
overall_rate = df["is_declining_label"].mean()
print(f"Overall decline rate: {overall_rate:.3f}")
print(
    "VERDICT: OPPOSITE — stale pages (n=174) decline LESS often (47.1%) than fresh pages "
    "(n=29,826, 54.2%), not more. Raw staleness alone points the wrong way in this slice."
)


Signal 1 — staleness (>=180d since update) vs. decline rate:
                     n  decline_rate
stale_bucket                        
fresh_under_180  29826      0.542480
stale_180plus      174      0.471264

Overall decline rate: 0.542
VERDICT: OPPOSITE — stale pages (n=174) decline LESS often (47.1%) than fresh pages (n=29,826, 54.2%), not more. Raw staleness alone points the wrong way in this slice.


In [3]:
# --- Signal 2: CTR vs. position tier (behind the CTR-fix logic), visible pages only ---
visible = df[df["impressions_90d"] >= 500]
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
signal2 = visible.groupby("position_tier")["ctr"].agg(n="size", mean_ctr="mean").reindex(tier_order)
print("Signal 2 — position tier vs. mean CTR (impressions_90d >= 500):")
print(signal2)
print()
ctr_values = signal2["mean_ctr"].dropna().values
monotonic_decreasing = all(ctr_values[i] >= ctr_values[i + 1] for i in range(len(ctr_values) - 1))
print(f"Monotonic decreasing top_3 -> deep: {monotonic_decreasing}")
print(
    "VERDICT: CONFIRMED — mean CTR falls cleanly from 34.7% (top_3, n=458) to 4.3% "
    "(deep, n=389) as position gets worse. CTR-for-position is a real pattern here, so "
    "'low CTR for its tier' is a legitimate, fixable anomaly worth flagging."
)


Signal 2 — position tier vs. mean CTR (impressions_90d >= 500):
                  n  mean_ctr
position_tier                
top_3           458  0.346572
page_1         7064  0.338808
striking       4485  0.266798
page_3_5       4330  0.143236
deep            389  0.043213

Monotonic decreasing top_3 -> deep: True
VERDICT: CONFIRMED — mean CTR falls cleanly from 34.7% (top_3, n=458) to 4.3% (deep, n=389) as position gets worse. CTR-for-position is a real pattern here, so 'low CTR for its tier' is a legitimate, fixable anomaly worth flagging.


**What the audit changed about my rule (this is the point of checking first):**

Signal 1 came back **OPPOSITE** of what a refresh flag assumes — so raw staleness is **excluded**
from both the score and the reason codes below. (I also tried the compound `stale AND visible`
condition the starter baseline uses: it *did* point the right direction, decline rate 94.1% vs.
54.2%, but on only **n=17** pages — too thin to build a rule on. Noted, not used.) That's the
negative that saved the rule from leaning on a signal that doesn't hold up here.

Signal 2 came back **CONFIRMED** with real n behind it, so it's the core of the rule.

**The rule, in plain words:** *"Review pages that have real search demand and are getting
noticeably fewer clicks than other pages ranking in the same position band — that's a fixable
CTR-friction problem (title, meta, snippet), not just noise. If a visible page is also unusually
thin, flag it for expansion instead."* Two conditions, checked in priority order, so every page
gets exactly **one** reason code and **one** action:

1. `demand_with_ctr_gap` → **refresh_and_review_ctr** — real demand (`impressions_90d >= 500`),
   ranked in the top 20 (`0 < avg_position <= 20`), and CTR under half the tier's typical CTR.
2. `thin_visible_page` → **expand_and_refresh** — has a recorded word count under 1,200 words
   and moderate demand (`impressions_90d >= 250`).
3. else `general_review` → **monitor** — nothing flagged; still ranked, lowest priority.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


In [4]:
def percentile_rank(series: pd.Series) -> pd.Series:
    return series.rank(method="average", pct=True).fillna(0)

# Tier-typical CTR, computed on visible pages only (same slice as the signal-2 check above).
tier_median_ctr = visible.groupby("position_tier")["ctr"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_median_ctr).fillna(0)
df["ctr_gap"] = (df["tier_median_ctr"] - df["ctr"]).clip(lower=0)

df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["ctr_gap_score"] = percentile_rank(df["ctr_gap"])

# word_count is missing for ~28% of 'keyword article' rows (not zero content) -- the
# flyrank-data skill's missingness trap. A blind fillna(0) would rank every missing-word-count
# row as maximally thin. Instead: score depth only where word_count is actually recorded.
df["has_word_count"] = df["word_count"].notna() & (df["word_count"] > 0)
wc_percentile = pd.Series(0.0, index=df.index)
wc_percentile.loc[df["has_word_count"]] = percentile_rank(df.loc[df["has_word_count"], "word_count"])
df["depth_gap_score"] = np.where(df["has_word_count"], (1 - wc_percentile) * df["visibility_score"], 0.0)

# ONE rule, three inputs, all pre-decision / observable -- no trend_direction, no trend_pct.
df["baseline_action_score"] = (
    0.45 * df["visibility_score"]
    + 0.40 * df["ctr_gap_score"]
    + 0.15 * df["depth_gap_score"]
).clip(0, 1)


def reason_and_action(row: pd.Series) -> tuple[str, str]:
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5 * row["tier_median_ctr"]:
        return "demand_with_ctr_gap", "refresh_and_review_ctr"
    if row["has_word_count"] and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        return "thin_visible_page", "expand_and_refresh"
    return "general_review", "monitor"


reasons = df.apply(reason_and_action, axis=1)
df["reason_code"] = [r[0] for r in reasons]
df["suggested_action"] = [r[1] for r in reasons]
df["rank"] = df["baseline_action_score"].rank(method="first", ascending=False).astype(int)

print("Reason code counts:")
print(df["reason_code"].value_counts())
print()
print("Action counts:")
print(df["suggested_action"].value_counts())


Reason code counts:
reason_code
general_review         26894
demand_with_ctr_gap     3029
thin_visible_page         77
Name: count, dtype: int64

Action counts:
suggested_action
monitor                   26894
refresh_and_review_ctr     3029
expand_and_refresh           77
Name: count, dtype: int64


In [5]:
output_columns = [
    "content_id", "client_id", "rank", "baseline_action_score",
    "visibility_score", "ctr_gap_score", "depth_gap_score",
    "reason_code", "suggested_action",
    "impressions_90d", "clicks_90d", "avg_position", "ctr", "tier_median_ctr",
    "word_count", "days_since_last_update", "content_age_days",
    "is_declining_label", "trend_direction",
]
queue = df[output_columns].sort_values("rank").reset_index(drop=True)

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)
print(f"Wrote {len(queue):,} ranked rows to {output_path}")

# Honest evaluation next to the base rate (building-baselines skill: never report P@K alone).
base_rate = df["is_declining_label"].mean()
precision_report = {f"precision_at_{k}": precision_at_k(df["is_declining_label"], df["baseline_action_score"], k) for k in (10, 20, 50)}
for k, p in precision_report.items():
    print(f"{k}: {p:.3f}  (base rate {base_rate:.3f})")


Wrote 30,000 ranked rows to work/outputs/baseline_action_score.csv
precision_at_10: 0.500  (base rate 0.542)
precision_at_20: 0.550  (base rate 0.542)
precision_at_50: 0.620  (base rate 0.542)


**Reading those numbers honestly:** Precision@10 and @20 sit at/near the base rate; only
Precision@50 clears it. That's expected, not a bug in the rule — `is_declining_label` is a
*current-window* trend proxy (W02's caveat), while this rule targets a different, narrower thing:
pages with a **fixable CTR-friction opportunity**. Those aren't the same target, so scoring this
rule against the decline proxy undersells it. The fairer eval (a CTR-recovery outcome label) is
a Week-5+ problem — noted here, not solved here.


## 3. Top-10 review

*For each of your top ten: action, why it's there, and what would make it wrong.*


In [6]:
top10 = queue.head(10).copy()
top10["ctr_vs_tier"] = (top10["ctr"] / top10["tier_median_ctr"]).round(2)
print(f"Top-10 client spread: {top10['client_id'].nunique()} distinct clients across 10 rows")
top10[["rank", "content_id", "client_id", "baseline_action_score", "reason_code",
       "suggested_action", "impressions_90d", "avg_position", "ctr", "tier_median_ctr",
       "ctr_vs_tier", "is_declining_label"]]


Top-10 client spread: 5 distinct clients across 10 rows


,rank,content_id,client_id,baseline_action_score,reason_code,suggested_action,impressions_90d,avg_position,ctr,tier_median_ctr,ctr_vs_tier,is_declining_label
0,1,content_d274ac4158ef,client_4e07408562,0.921413,demand_with_ctr_gap,refresh_and_review_ctr,65138,6.8,0.01,0.24,0.04,0
1,2,content_b49efa4db88a,client_3fdba35f04,0.912363,demand_with_ctr_gap,refresh_and_review_ctr,46866,4.6,0.03,0.24,0.12,0
2,3,content_9c8299b55f3c,client_624b60c58c,0.904631,demand_with_ctr_gap,refresh_and_review_ctr,54783,8.5,0.03,0.24,0.12,0
3,4,content_339b357d04c7,client_bbb965ab0c,0.894803,demand_with_ctr_gap,refresh_and_review_ctr,46879,3.7,0.01,0.24,0.04,0
4,5,content_453722754fea,client_f369cb89fc,0.887139,demand_with_ctr_gap,refresh_and_review_ctr,140079,7.6,0.01,0.24,0.04,1
5,6,content_ca17a024f90c,client_4e07408562,0.883460,demand_with_ctr_gap,refresh_and_review_ctr,38815,9.1,0.01,0.24,0.04,1
6,7,content_39881853ef0c,client_f369cb89fc,0.875132,demand_with_ctr_gap,refresh_and_review_ctr,112434,7.2,0.01,0.24,0.04,1
7,8,content_0919dd345d80,client_4e07408562,0.872211,demand_with_ctr_gap,refresh_and_review_ctr,119217,7.0,0.02,0.24,0.08,1
8,9,content_865fd632a1f3,client_f369cb89fc,0.871792,demand_with_ctr_gap,refresh_and_review_ctr,39246,6.1,0.03,0.24,0.12,1
9,10,content_c84a0ab98e90,client_f369cb89fc,0.870017,demand_with_ctr_gap,refresh_and_review_ctr,223271,7.8,0.03,0.24,0.12,0


In [7]:
for _, row in top10.iterrows():
    why = (
        f"impressions_90d={row['impressions_90d']:,.0f}, avg_position={row['avg_position']:.1f}, "
        f"ctr={row['ctr']:.2f}% vs. tier-typical {row['tier_median_ctr']:.2f}% "
        f"({row['ctr_vs_tier']:.0%} of what pages at this position usually get)."
    )
    if row["reason_code"] == "demand_with_ctr_gap":
        wrong_if = (
            "wrong if this low CTR is a SERP-feature artifact (featured snippet / zero-click "
            "result eating the click) rather than a title/meta problem -- a copy refresh "
            "wouldn't move a snippet-suppressed CTR."
        )
    else:
        wrong_if = (
            "wrong if the low word count reflects an intentionally short format (e.g. a quick "
            "answer page) rather than thin content -- expanding it could hurt, not help."
        )
    label_note = "matches the decline proxy" if row["is_declining_label"] == 1 else "NOT flagged by the decline proxy -- worth a second look before trusting rank alone"
    print(f"#{int(row['rank'])} [{row['content_id']}] -> {row['suggested_action']} ({row['reason_code']})")
    print(f"   why: {why}")
    print(f"   what would make it wrong: {wrong_if}")
    print(f"   decline-proxy check: {label_note}")
    print()


#1 [content_d274ac4158ef] -> refresh_and_review_ctr (demand_with_ctr_gap)
   why: impressions_90d=65,138, avg_position=6.8, ctr=0.01% vs. tier-typical 0.24% (4% of what pages at this position usually get).
   what would make it wrong: wrong if this low CTR is a SERP-feature artifact (featured snippet / zero-click result eating the click) rather than a title/meta problem -- a copy refresh wouldn't move a snippet-suppressed CTR.
   decline-proxy check: NOT flagged by the decline proxy -- worth a second look before trusting rank alone

#2 [content_b49efa4db88a] -> refresh_and_review_ctr (demand_with_ctr_gap)
   why: impressions_90d=46,866, avg_position=4.6, ctr=0.03% vs. tier-typical 0.24% (12% of what pages at this position usually get).
   what would make it wrong: wrong if this low CTR is a SERP-feature artifact (featured snippet / zero-click result eating the click) rather than a title/meta problem -- a copy refresh wouldn't move a snippet-suppressed CTR.
   decline-proxy check: NOT f

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


In [8]:
# --- Leakage check: assert the label-derived / product-decision columns never entered the rule ---
scoring_inputs = {
    "visibility_score", "ctr_gap_score", "depth_gap_score", "baseline_action_score",
    "impressions_90d", "avg_position", "ctr", "tier_median_ctr", "word_count", "has_word_count",
}
forbidden = {"trend_direction", "trend_pct", "is_declining_label",
             "health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win"}
leaked = scoring_inputs & forbidden
assert not leaked, f"Leakage: {leaked} used as a scoring input"
print("Leakage check passed: no label-derived column or FlyRank product flag feeds the score,")
print("the reason code, or the action. trend_direction/trend_pct/is_declining_label appear in")
print("the output CSV only as an audit column, never as a rule input (confirmed in section 2's")
print("scoring code above -- reason_and_action() never reads them).")


Leakage check passed: no label-derived column or FlyRank product flag feeds the score,
the reason code, or the action. trend_direction/trend_pct/is_declining_label appear in
the output CSV only as an audit column, never as a rule input (confirmed in section 2's
scoring code above -- reason_and_action() never reads them).


**Weak picks, found by looking hard at the queue (not just the top 10):**

1. **Client concentration risk.** Before I fixed the `word_count` missingness bug below, 8 of the
   top 10 rows belonged to a single client (`client_19581e27de`) purely because that client has a
   handful of pages with six-figure impression counts — `visibility_score` alone can let one
   high-traffic client crowd out everyone else's real opportunities. The current top 10 (§3) is
   more spread (`top10['client_id'].nunique()` distinct clients), but the queue has no per-client
   cap, so a future pass should add one before this goes to a real reviewer.
2. **The `word_count` NaN trap (caught, then fixed).** My first version of `depth_gap_score` used
   a plain `fillna(0)` on `word_count`, and `word_count` is missing for ~28% of `keyword article`
   rows (not zero-length — just unmeasured, per the flyrank-data skill's missingness warning).
   That silently scored every missing-word-count row as maximally "thin," and every top-10 row
   was a `keyword article` with `word_count = NaN` as a direct result. Fixed with a
   `has_word_count` flag (§2) instead of imputing — this is the exact "blind fillna(0) injects a
   category signal" trap the skill names, caught on my own rule rather than read about.
3. **`general_review` / `monitor` is 88% of the queue** (26,894 of 30,000 pages) — the rule is
   deliberately conservative: only a real CTR gap or thin+visible content earns a stronger action.
   That's a feature for a first pass (fewer false alarms) but means most of the "worth reviewing"
   judgment calls from W01/W02 aren't captured yet — a Week-5 model has real room to add value
   beyond these two hand-written conditions.


In [9]:
import json as _json

metrics = {
    "rows": int(len(queue)),
    "base_rate": float(base_rate),
    "precision": precision_report,
    "reason_code_counts": df["reason_code"].value_counts().to_dict(),
    "action_counts": df["suggested_action"].value_counts().to_dict(),
    "signal_verdicts": {
        "staleness_vs_decline": {
            "verdict": "OPPOSITE",
            "fresh_under_180_n": int(signal1.loc["fresh_under_180", "n"]),
            "fresh_under_180_decline_rate": float(signal1.loc["fresh_under_180", "decline_rate"]),
            "stale_180plus_n": int(signal1.loc["stale_180plus", "n"]),
            "stale_180plus_decline_rate": float(signal1.loc["stale_180plus", "decline_rate"]),
        },
        "ctr_vs_position": {
            "verdict": "CONFIRMED",
            "monotonic_decreasing_top3_to_deep": bool(monotonic_decreasing),
            "mean_ctr_by_tier": signal2["mean_ctr"].round(4).to_dict(),
            "n_by_tier": signal2["n"].dropna().astype(int).to_dict(),
        },
    },
    "score_formula": {"visibility_score": 0.45, "ctr_gap_score": 0.40, "depth_gap_score": 0.15},
    "top10_distinct_clients": int(top10["client_id"].nunique()),
}
metrics_path = Path("work/outputs/w04_baseline_metadata.json")
metrics_path.write_text(_json.dumps(metrics, indent=2, sort_keys=True))
print(f"Wrote {metrics_path}")


Wrote work/outputs/w04_baseline_metadata.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
